# 02 · Draft win predictor
**Question:** given the 12 heroes and the average rank, how likely is the Hidden King side to win?
**Inputs:** `h_<id>` = +1 on team 0, -1 on team 1; `badge` = average badge / 10. **Label:** team 0 won.
**Split:** by time (train old, test new). **Report:** AUC, log loss, Brier, ECE, reliability plot, against two baselines.
Expect modest numbers: drafts alone usually give an AUC in the mid 0.5s to low 0.6s in hero games. Calibration is what matters for a "54% favoured" widget.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from broadway_ml.flatten import load_tables, item_catalog
from broadway_ml import features as fe, evaluate as ev, export as ex
T = load_tables('../data/tables')
{k: len(v) for k, v in T.items()}

In [ ]:
X, y, heroes, mids = fe.draft_features(T['matches'], T['players'])
start = T['matches'].set_index('match_id').loc[mids, 'start_time'].to_numpy()
tr, te = ev.time_split(start, test_frac=0.2)
print(X.shape, 'train', tr.sum(), 'test', te.sum(), 'base rate', y[tr].mean().round(4))

## Baselines
1. **Base rate:** always predict the training win rate of team 0.
2. **Hero win rates:** sum of each hero's log-odds of winning (from training data), team 0 minus team 1. No interactions, no fitting beyond counting.

In [ ]:
def hero_logodds(X, y):
    H = X.filter(like='h_')
    on0, on1 = (H == 1), (H == -1)
    wins = on0.mul(y, axis=0).sum() + on1.mul(1 - y, axis=0).sum()
    games = on0.sum() + on1.sum()
    wr = (wins + 10) / (games + 20)  # shrink rare heroes toward 50%
    return np.log(wr / (1 - wr))
lo = hero_logodds(X[tr], y[tr])
z = X[te].filter(like='h_').mul(lo).sum(axis=1).to_numpy()
preds = {'base rate': np.full(te.sum(), y[tr].mean()), 'hero win rates': 1 / (1 + np.exp(-z))}
ev.compare(y[te], preds)

## Your model
Starting point: L2 logistic regression on the same features (the export format supports exactly this). Ideas to try, roughly in order of payoff:
- tune `C` with a time-ordered validation fold (not random K-fold)
- `fe.mirror_draft` to remove side bias, and check whether side matters at all
- hero-pair synergy features for teammates and counter features across teams (many columns; needs strong regularisation)
- badge interactions (some heroes are stronger at high rank)
- calibrate with `CalibratedClassifierCV(method='sigmoid')` on a later time slice if ECE is poor

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
# TODO(V): this is the plain starting point; replace or extend it
scaler = StandardScaler().fit(X[tr])
model = LogisticRegression(C=0.1, max_iter=2000).fit(scaler.transform(X[tr]), y[tr])
preds['logistic'] = model.predict_proba(scaler.transform(X[te]))[:, 1]
report = ev.compare(y[te], preds); report

In [ ]:
ev.plot_reliability(y[te], {k: v for k, v in preds.items() if k != 'base rate'});
ev.reliability_table(y[te], preds['logistic'])

In [ ]:
# Strongest heroes according to the model (positive = helps the side it is on)
pd.Series(model.coef_[0][:-1] / scaler.scale_[:-1], index=X.columns[:-1]).sort_values().iloc[[*range(5), *range(-5, 0)]]

## Export
Only export when the model beats both baselines on log loss and the reliability plot hugs the diagonal.

In [ ]:
ex.export_linear(model, list(X.columns), '../models/draft_win.json', scaler=scaler, name='draft-win',
                 metrics=report.loc['logistic'].to_dict(),
                 notes=f'trained on {tr.sum()} matches up to {pd.to_datetime(start[tr].max(), unit="s"):%Y-%m-%d}')